# ChemicalEntity → BiologicalProcess Relation Pipeline

Builds a unified, deduplicated edge table for the **ChemicalEntity–BiologicalProcess** relation.\n*Note: This strictly uses natively defined Chemical->Biological edges and includes kg_type classification.*

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | head_id_is | tail_id_is | head_detail_name | tail_detail_name | kg_type`

---
**Sources included:**
- Pheknowlator\n- AgeAnnoMO

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [40]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_BIOLOGICALPROCESS/ALL_CHEMICALENTITY_BIOLOGICALPROCESS.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name', 'species'
]

## 1 · Mapping DBs

In [8]:
Pubchem = pd.read_pickle(MAPPING_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem

GO = pd.read_csv(MAPPING_DIR + 'MESH/MESH_GO_ID_NAME.csv')
GO_dict = dict(zip(GO['id'], GO['name']))
del GO

## 2 · Load Source Files

In [44]:
# Pheknowlator (Native CE -> BP)
Pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Chemical_BiologicalProcess.csv')
Pheknowlator.rename(columns={'Head_IUPAC': 'head_detail_name'}, inplace=True)
Pheknowlator.columns = Pheknowlator.columns.str.lower()
Pheknowlator['head_id_is'] = 'PubChem'
Pheknowlator['tail_id_is'] = 'Quick_GO'
Pheknowlator['relation'] = 'ChemicalEntity_BiologicalProcess'
Pheknowlator['head_type'] = 'ChemicalEntity'
Pheknowlator['tail_type'] = 'BiologicalProcess'
Pheknowlator['kg_source'] = 'pheknowlator'
Pheknowlator['kg_type'] = 'Generalised'
AgeAnnoMO['species'] = 'Homo sapiens'
Pheknowlator["head"] = Pheknowlator["head"].astype(str).str.replace(r'\.0$', '', regex=True)
Pheknowlator["head"] = Pheknowlator["head"].str.replace('PUBCHEM:', '', regex=False)

print(f"Pheknowlator CE→BP: {len(Pheknowlator):,} rows")
Pheknowlator

Pheknowlator CE→BP: 12,950 rows


,head,relation,tail,head_type,tail_type,head_detail_name,head_smiles,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,24502,ChemicalEntity_BiologicalProcess,GO:0042964,ChemicalEntity,BiologicalProcess,dipotassium;oxido-(oxido(dioxo)chromio)oxy-dio...,[O-][Cr](=O)(=O)O[Cr](=O)(=O)[O-].[K+].[K+],obsolete thioredoxin reduction,PubChem,Quick_GO,pheknowlator,Generalised
1,6758,ChemicalEntity_BiologicalProcess,GO:0010800,ChemicalEntity,BiologicalProcess,"(1S,6R,13S)-16,17-dimethoxy-6-prop-1-en-2-yl-2...",CC(=C)[C@H]1CC2=C(O1)C=CC3=C2O[C@@H]4COC5=CC(=...,positive regulation of peptidyl-threonine phos...,PubChem,Quick_GO,pheknowlator,Generalised
2,774,ChemicalEntity_BiologicalProcess,GO:0006915,ChemicalEntity,BiologicalProcess,2-(1H-imidazol-5-yl)ethanamine,C1=C(NC=N1)CCN,apoptotic process,PubChem,Quick_GO,pheknowlator,Generalised
3,433294,ChemicalEntity_BiologicalProcess,GO:0006915,ChemicalEntity,BiologicalProcess,lithium;chloride,[Li+].[Cl-],apoptotic process,PubChem,Quick_GO,pheknowlator,Generalised
4,5113032,ChemicalEntity_BiologicalProcess,GO:0010942,ChemicalEntity,BiologicalProcess,"6-chloro-2,3,4,9-tetrahydro-1H-carbazole-1-car...",C1CC(C2=C(C1)C3=C(N2)C=CC(=C3)Cl)C(=O)N,obsolete positive regulation of cell death,PubChem,Quick_GO,pheknowlator,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...
12945,896,ChemicalEntity_BiologicalProcess,GO:0008284,ChemicalEntity,BiologicalProcess,N-[2-(5-methoxy-1H-indol-3-yl)ethyl]acetamide,CC(=O)NCCC1=CNC2=C1C=C(C=C2)OC,positive regulation of cell population prolife...,PubChem,Quick_GO,pheknowlator,Generalised
12946,443495,ChemicalEntity_BiologicalProcess,GO:0008219,ChemicalEntity,BiologicalProcess,sodium;oxoarsinite,[O-][As]=O.[Na+],cell death,PubChem,Quick_GO,pheknowlator,Generalised
12947,122164822,ChemicalEntity_BiologicalProcess,GO:0008283,ChemicalEntity,BiologicalProcess,hydroxy(oxo)iron;nickel,O[Fe]=O.O[Fe]=O.[Ni],cell population proliferation,PubChem,Quick_GO,pheknowlator,Generalised
12948,20469,ChemicalEntity_BiologicalProcess,GO:0046649,ChemicalEntity,BiologicalProcess,"(8S,9R,10S,11S,13S,14S,16S,17R)-9-chloro-11,17...",C[C@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@@]4([C@]...,lymphocyte activation,PubChem,Quick_GO,pheknowlator,Generalised


In [45]:
# AgeAnnoMO (Native CE -> BP)
AgeAnnoMO = pd.read_csv(PROC_DIR + 'ageanno_processed_mo/AgeAnnoMO_Human_Chemical_BiologicalProcess_2.csv')
AgeAnnoMO.columns = AgeAnnoMO.columns.str.lower()

AgeAnnoMO['head_id_is'] = 'PubChem'
AgeAnnoMO['tail_id_is'] = 'Quick_GO'
AgeAnnoMO['tail_type'] = 'BiologicalProcess'
AgeAnnoMO['kg_source'] = 'AgeAnnoMO'
AgeAnnoMO['kg_type'] = 'Aging'
AgeAnnoMO['species'] = 'Homo sapiens'
AgeAnnoMO["head"] = AgeAnnoMO["head"].astype(str).str.replace(r'\.0$', '', regex=True)

print(f"AgeAnnoMO CE→BP: {len(AgeAnnoMO):,} rows")
AgeAnnoMO

AgeAnnoMO CE→BP: 17 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source,species,head_id,kg_source,kg_type
0,80220,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"1-methyl-3,7-dihydropurine-2,6-dione",Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,1-Methylxanthine,AgeAnnoMO,Aging
1,700653,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(2S)-2-acetamido-3-(1H-indol-3-yl)propanoic acid,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,N-Acetyl-L-tryptophan,AgeAnnoMO,Aging
2,464,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,2-benzamidoacetic acid,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,Hippuric acid,AgeAnnoMO,Aging
3,6262,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"(2S)-2,5-diaminopentanoic acid",Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,L-ornithine,AgeAnnoMO,Aging
4,7405,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(2S)-5-oxopyrrolidine-2-carboxylic acid,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,L-Pyroglutamic acid,AgeAnnoMO,Aging
5,7018721,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(2S)-2-amino-5-[[(2S)-1-(carboxymethylamino)-1...,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,Ophthalmic acid,AgeAnnoMO,Aging
6,588,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,2-amino-3-methyl-4H-imidazol-5-one,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,2-Imino-1-methylimidazolidin-4-one,AgeAnnoMO,Aging
7,611,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,2-aminopentanedioic acid,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,DL-Glutamic acid,AgeAnnoMO,Aging
8,168379,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(3R)-3-(2-methylpropanoyloxy)-4-(trimethylazan...,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,Isobutyryl-L-carnitine,AgeAnnoMO,Aging
9,6426851,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,3-(3-methylbutanoyloxy)-4-(trimethylazaniumyl)...,Deregulated Nutrient Sensing,PubChem,Quick_GO,AgeAnnoMO,Homo sapiens,Isovalerylcarnitine,AgeAnnoMO,Aging


In [46]:
metaboage = pd.read_csv(PROC_DIR + 'MetaboAge/MetaboAge_Human_Chem_BioPro_direct.csv')
metaboage.columns = metaboage.columns.str.lower()

metaboage['head_id_is'] = 'PubChem'
metaboage['tail_id_is'] = 'Quick_GO'
metaboage['tail_type'] = 'BiologicalProcess'
metaboage['kg_source'] = 'MetaboAge'
metaboage['kg_type'] = 'Aging'
metaboage['species'] = 'Homo sapiens'

metaboage["head"] = metaboage["head"].astype(str).str.replace(r'\.0$', '', regex=True)
print(f"metaboage CE→BP: {len(metaboage):,} rows")
metaboage

metaboage CE→BP: 1,519 rows


,head,relation,tail,head_type,tail_type,relation_source,species,head_alt_name,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,3845,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,kynurenic acid,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
1,3845,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,kynurenic acid,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
2,5280450,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,linoleic acid,"(9Z,12Z)-octadeca-9,12-dienoic acid",CCCCC/C=C\C/C=C\CCCCCCCC(=O)O,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
3,5280450,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,linoleic acid,"(9Z,12Z)-octadeca-9,12-dienoic acid",CCCCC/C=C\C/C=C\CCCCCCCC(=O)O,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
4,124886,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,glutathione,(2S)-2-amino-5-[[(2R)-1-(carboxymethylamino)-1...,C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1514,5571,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,N-methylnicotinic acid,1-methylpyridin-1-ium-3-carboxylic acid,C[N+]1=CC=CC(=C1)C(=O)O,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
1515,440810,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,N-methyl-4-pyridone-3-carboxamide,1-methyl-4-oxopyridine-3-carboxamide,CN1C=CC(=O)C(=C1)C(=O)N,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
1516,440810,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,N-methyl-4-pyridone-3-carboxamide,1-methyl-4-oxopyridine-3-carboxamide,CN1C=CC(=O)C(=C1)C(=O)N,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging
1517,54679076,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,ascorbate,"(2R)-2-[(1S)-1,2-dihydroxyethyl]-3-hydroxy-5-o...",C([C@@H]([C@@H]1C(=C(C(=O)O1)[O-])O)O)O,obsolete aging,PubChem,Quick_GO,MetaboAge,Aging


## 3 · Consolidate

In [47]:
source_dfs = [Pheknowlator, AgeAnnoMO, metaboage]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,24502,ChemicalEntity_BiologicalProcess,GO:0042964,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,dipotassium;oxido-(oxido(dioxo)chromio)oxy-dio...,obsolete thioredoxin reduction,None
1,6758,ChemicalEntity_BiologicalProcess,GO:0010800,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,"(1S,6R,13S)-16,17-dimethoxy-6-prop-1-en-2-yl-2...",positive regulation of peptidyl-threonine phos...,None
2,774,ChemicalEntity_BiologicalProcess,GO:0006915,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,2-(1H-imidazol-5-yl)ethanamine,apoptotic process,None
3,433294,ChemicalEntity_BiologicalProcess,GO:0006915,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,lithium;chloride,apoptotic process,None
4,5113032,ChemicalEntity_BiologicalProcess,GO:0010942,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,"6-chloro-2,3,4,9-tetrahydro-1H-carbazole-1-car...",obsolete positive regulation of cell death,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14481,5571,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,MetaboAge,Aging,PubChem,Quick_GO,1-methylpyridin-1-ium-3-carboxylic acid,obsolete aging,Homo sapiens
14482,440810,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,MetaboAge,Aging,PubChem,Quick_GO,1-methyl-4-oxopyridine-3-carboxamide,obsolete aging,Homo sapiens
14483,440810,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,MetaboAge,Aging,PubChem,Quick_GO,1-methyl-4-oxopyridine-3-carboxamide,obsolete aging,Homo sapiens
14484,54679076,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,None,BiologicalProcess,MetaboAge,Aging,PubChem,Quick_GO,"(2R)-2-[(1S)-1,2-dihydroxyethyl]-3-hydroxy-5-o...",obsolete aging,Homo sapiens


In [48]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,12141,0,12141
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [49]:
final_df[final_df['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
40,7316,ChemicalEntity_BiologicalProcess,GO:0034374,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,low-density lipoprotein particle remodeling,None
43,7316,ChemicalEntity_BiologicalProcess,GO:1903409,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,reactive oxygen species biosynthetic process,None
95,678,ChemicalEntity_BiologicalProcess,GO:0008283,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,cell population proliferation,None
188,3032552,ChemicalEntity_BiologicalProcess,GO:0060348,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,bone development,None
389,1615449,ChemicalEntity_BiologicalProcess,GO:0034976,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,response to endoplasmic reticulum stress,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12794,7316,ChemicalEntity_BiologicalProcess,GO:0010729,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,positive regulation of hydrogen peroxide biosy...,None
12839,1615449,ChemicalEntity_BiologicalProcess,GO:0034599,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,cellular response to oxidative stress,None
12881,1230607,ChemicalEntity_BiologicalProcess,GO:0051881,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,regulation of mitochondrial membrane potential,None
12892,3032552,ChemicalEntity_BiologicalProcess,GO:0008217,ChemicalEntity,None,BiologicalProcess,pheknowlator,Generalised,PubChem,Quick_GO,NaN,regulation of blood pressure,None


## 4 · Chemical Name Normalisation

In [50]:
head_ids = pd.to_numeric(final_df['head'], errors='coerce')
mask = final_df['head_detail_name'].isna()
final_df.loc[mask, 'head_detail_name'] = head_ids[mask].map(Pubchem_IUPAC_CID_Dict)

# # Also fill tail detail if missing
# tail_detail = final_df['tail'].map(GO_dict)
# mask = final_df['tail_detail_name'].isna()
# final_df.loc[mask, 'tail_detail_name'] = tail_detail[mask]

final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)
print(f"Consolidated rows: {len(final_df):,}")

Consolidated rows: 11,979


In [51]:
nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,11979,0,11979
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


## 5 · Deduplication

In [52]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})
print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")

Before dedup: 11,979  |  After dedup: 11,098


## 6 · Save Output

In [53]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 11,098 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_BIOLOGICALPROCESS/ALL_CHEMICALENTITY_BIOLOGICALPROCESS.csv
